# Stage 1 Kaggle Run-All Notebook

This notebook is the Kaggle-first entry point for Stage 1. Attach the dataset in Kaggle, then run all cells.

It will auto-clone the repository if source files are missing, resolve the dataset mount, run bootstrap checks, start Stage 1 training, and print artifact status.

## Parameters

In [ ]:
# Parameters
HARDWARE_PROFILE = "auto"   # options: "auto", "p100", "t4x2"
RUN_MODE = "first"          # options: "first" or "resume"
REPO_URL = "https://github.com/HoangNguyennnnnnn/DoAn_backup.git"
DATASET_SLUG = "balraj98/modelnet40-princeton-3d-object-dataset"
DATASET_ROOT_OVERRIDE = ""   # optional explicit /kaggle/input path
RESUME_CHECKPOINT_PATH = ""   # leave empty to use latest_step.ckpt in resume mode
RUN_ID = "kaggle-stage1"
DECODE_NUM_SAMPLES = 4
DECODE_THRESHOLD = 0.5

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path


def _run(command, cwd=None):
    printable = " ".join(str(part) for part in command)
    print(f"$ {printable}")
    subprocess.run([str(part) for part in command], cwd=str(cwd) if cwd else None, check=True)


def _ensure_repo_root(repo_url: str) -> Path:
    repo_name = Path(repo_url.rstrip("/")).name.replace(".git", "")
    candidates = [
        Path("/kaggle/working/DoAn_backup"),
        Path("/kaggle/working") / repo_name,
        Path.cwd(),
    ]

    for candidate in candidates:
        if (candidate / "scripts").is_dir() and (candidate / "configs").is_dir():
            if (candidate / ".git").exists():
                subprocess.run(["git", "pull", "--ff-only"], cwd=str(candidate), check=True)
            return candidate.resolve()

    target = Path("/kaggle/working/DoAn_backup")
    if target.exists():
        shutil.rmtree(target)
    _run(["git", "clone", repo_url, str(target)])
    return target.resolve()


def _find_dataset_root(dataset_slug: str, explicit_root: str = "") -> Path:
    def _is_modelnet_structure(path: Path) -> bool:
        """Check if path has ModelNet-style class directories with train/test."""
        if not path.is_dir():
            return False
        class_dirs = [item for item in path.iterdir() 
                      if item.is_dir() and item.name.lower() not in {"__macosx", ".ipynb_checkpoints"}]
        valid = sum(1 for cd in class_dirs if (cd / "train").is_dir() or (cd / "test").is_dir())
        return valid >= 5

    input_root = Path("/kaggle/input")
    candidates = []
    if explicit_root.strip():
        candidates.append(Path(explicit_root.strip()))
    candidates.append(input_root / dataset_slug.split("/")[-1])

    if input_root.exists():
        candidates.extend(child for child in sorted(input_root.iterdir()) if child.is_dir())

    seen = set()
    # First pass: check direct candidates
    for candidate in candidates:
        resolved = candidate.resolve() if candidate.exists() else candidate
        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)
        if resolved.exists() and _is_modelnet_structure(resolved):
            return resolved

    # Second pass: search recursively for ModelNet structure
    if input_root.exists():
        try:
            for root, dirs, files in os.walk(str(input_root)):
                if _is_modelnet_structure(Path(root)):
                    return Path(root)
        except Exception:
            pass

    mounted = [child.name for child in sorted(input_root.iterdir()) if child.is_dir()] if input_root.exists() else []
    raise RuntimeError(
        "Dataset root missing. Attach the Kaggle dataset in the notebook UI before training. "
        f"Checked candidates: {[str(candidate) for candidate in candidates]}. "
        f"Mounted /kaggle/input folders: {mounted or ['none']}"
    )


def _resolve_hardware_config(profile: str) -> Path:
    profile = profile.strip().lower()
    if profile == "p100":
        return Path("configs/hardware_p100.yaml")
    if profile == "t4x2":
        return Path("configs/hardware_t4x2.yaml")
    if profile != "auto":
        raise ValueError(f"Unsupported HARDWARE_PROFILE: {profile}")

    try:
        import torch

        if torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(0).lower()
            if "t4" in gpu_name and torch.cuda.device_count() >= 2:
                return Path("configs/hardware_t4x2.yaml")
    except Exception:
        pass

    return Path("configs/hardware_p100.yaml")


def _normalize_stage1_config(repo_root: Path) -> None:
    config_path = repo_root / "configs" / "train_stage1.yaml"
    if not config_path.exists():
        return

    text = config_path.read_text(encoding="utf-8")
    if "${GRAD_ACC_STEPS}" in text:
        config_path.write_text(text.replace("${GRAD_ACC_STEPS}", "1"), encoding="utf-8")
        print(f"Normalized placeholder in {config_path}")

In [ ]:
repo_root = _ensure_repo_root(REPO_URL)
os.chdir(repo_root)
print(f"Project root: {repo_root}")

_normalize_stage1_config(repo_root)

dataset_root = _find_dataset_root(DATASET_SLUG, DATASET_ROOT_OVERRIDE)
hardware_config = _resolve_hardware_config(HARDWARE_PROFILE)
output_root = Path("/kaggle/working")

os.environ["DATASET_SLUG"] = DATASET_SLUG
os.environ["DATASET_ROOT"] = str(dataset_root)
os.environ["OUTPUT_ROOT"] = str(output_root)
os.environ["TRAIN_CONFIG"] = "configs/train_stage1.yaml"
os.environ["DATA_CONFIG"] = "configs/data_stage1.yaml"
os.environ["HARDWARE_CONFIG"] = str(hardware_config)
os.environ["RUN_ID"] = RUN_ID

if RUN_MODE not in {"first", "resume"}:
    raise ValueError(f"Unsupported RUN_MODE: {RUN_MODE}")

if RUN_MODE == "resume":
    resume_path = RESUME_CHECKPOINT_PATH.strip() or str(output_root / "checkpoints" / "latest_step.ckpt")
    os.environ["RESUME_CHECKPOINT_PATH"] = resume_path

print(f"Dataset root: {dataset_root}")
print(f"Hardware config: {hardware_config}")
print(f"Output root: {output_root}")
if RUN_MODE == "resume":
    print(f"Resume checkpoint: {os.environ['RESUME_CHECKPOINT_PATH']}")

_run(["bash", "scripts/kaggle_bootstrap.sh"], cwd=repo_root)

In [ ]:
train_cmd = [
    "python",
    "scripts/train_stage1.py",
    "--config",
    os.environ["TRAIN_CONFIG"],
    "--hardware",
    os.environ["HARDWARE_CONFIG"],
    "--data-config",
    os.environ["DATA_CONFIG"],
    "--dataset-root",
    os.environ["DATASET_ROOT"],
    "--output-root",
    os.environ["OUTPUT_ROOT"],
    "--device",
    "auto",
    "--run-id",
    os.environ["RUN_ID"],
    "--autoresume",
    "--contract-smoke",
]

if RUN_MODE == "resume":
    train_cmd.extend(["--resume-from", os.environ["RESUME_CHECKPOINT_PATH"]])

_run(train_cmd, cwd=repo_root)

In [ ]:
checkpoint_dir = Path(os.environ["OUTPUT_ROOT"]) / "checkpoints"
log_dir = Path(os.environ["OUTPUT_ROOT"]) / "logs"

print("Checkpoint status:")
for name in ["latest_step.ckpt", "interrupt.ckpt", "latest.ckpt", "best.ckpt"]:
    path = checkpoint_dir / name
    print(f"  {name}: exists={path.exists()} path={path}")

print("\nDecode sanity reports:")
for name in ["stage1_decode_sanity_report.json", "stage1_decode_sanity_report.md"]:
    path = log_dir / name
    print(f"  {name}: exists={path.exists()} path={path}")

metrics_path = log_dir / "stage1_training_metrics.jsonl"
print(f"\nMetrics log exists={metrics_path.exists()} path={metrics_path}")

summary_candidates = sorted((Path(os.environ["OUTPUT_ROOT"]) / "runs").glob("*/metadata/train_summary.json"))
if summary_candidates:
    latest_summary = summary_candidates[-1]
    print(f"\nLatest train summary: {latest_summary}")
    print(latest_summary.read_text(encoding="utf-8")[:2000])